# Cuaderno 04: Diseño, Evaluación y Entrenamiento de Modelos de ML
Este cuaderno cubre desde la formulación del problema y baselines hasta la evaluación en prueba, interpretación, ablación y prueba de humo del modelo guardado.

In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    brier_score_loss,
    classification_report,
    f1_score,
    precision_recall_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    learning_curve,
)

from flows.common.paths import BRONZE, BRONZE_UNLABELED_NAME, MODELS

ml = importlib.import_module("flows.05_ml_models")
de = importlib.import_module("flows.01_data_engineering")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.figsize'] = (8, 5)


## Parte 10 — Del análisis a la predicción: formulación del problema
### Paso 39 y 40: Carga del dataset y auditoría de variables

In [ ]:
X, y = ml.load_dataset()
print("Dimensiones de X:", X.shape)
print("Tasa de supervivencia global:", round(float(y.mean()), 4))
print("Columnas en X candidatas:", list(X.columns))
X.head()


### Paso 41: División estratificada entrenamiento / prueba
Se divide una única vez. `X_test` no se toca hasta la evaluación final en la Parte 16.

In [ ]:
X_train, X_test, y_train, y_test = ml.split(X, y)
print(f"Entrenamiento: {X_train.shape[0]} filas, tasa de supervivencia: {y_train.mean():.4f}")
print(f"Prueba: {X_test.shape[0]} filas, tasa de supervivencia: {y_test.mean():.4f}")


### Paso 42: Baselines a superar

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline 1: Dummy más frecuente
dummy_pipe = ml.build_pipeline(DummyClassifier(strategy='most_frequent'))
dummy_preds = cross_val_predict(dummy_pipe, X_train, y_train, cv=cv)
print("--- Baseline Ingenuo (Most Frequent) ---")
print(classification_report(y_train, dummy_preds, zero_division=0))

# Baseline 2: Regla de género (mujeres = 1, hombres = 0)
gender_preds = (X_train['Sex'] == 'female').astype(int)
print("--- Baseline Heurístico (Regla de Género) ---")
print(classification_report(y_train, gender_preds))


## Parte 11 — Preprocesamiento reproducible con `Pipeline`
### Paso 43: Demostración de Fuga de Datos (Data Leakage)

In [ ]:
media_mujeres_global = X.loc[X['Sex'] == 'female', 'Age'].mean()
media_mujeres_train = X_train.loc[X_train['Sex'] == 'female', 'Age'].mean()

print(f"Edad media mujeres (todos los datos X): {media_mujeres_global:.4f}")
print(f"Edad media mujeres (solo X_train):      {media_mujeres_train:.4f}")
print(f"Diferencia: {abs(media_mujeres_global - media_mujeres_train):.4f} años")


### Paso 44 a 46: Verificación del Pipeline y One-Hot Encoding

In [ ]:
pipe_check = ml.build_pipeline(DummyClassifier())
prep_transformer = pipe_check[:-1].fit(X_train)
Z_test = prep_transformer.transform(X_test)

print(f"¿Existen NaN en la matriz preprocesada? {np.isnan(Z_test).any()}")
print(f"Número de columnas tras preprocesamiento: {Z_test.shape[1]}")
feature_names = prep_transformer['prep'].get_feature_names_out()
print("Nombres de variables codificadas:")
print(feature_names)


## Parte 12 — Modelos candidatos y validación cruzada
### Pasos 47 a 50: Comparación de modelos candidatos

In [ ]:
cv_summary = ml.cross_validate_candidates(X_train, y_train)
display(cv_summary)


### Gráfico de desempeño y sobreajuste por modelo

In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(data=cv_summary, x='model', y='test_roc_auc_mean', color='steelblue')
plt.errorbar(x=range(len(cv_summary)), y=cv_summary['test_roc_auc_mean'],
             yerr=cv_summary['test_roc_auc_std'], fmt='none', c='black', capsize=5)
plt.title('ROC-AUC Promedio (+/- 1 std) en Validación Cruzada')
plt.ylabel('ROC-AUC')
plt.ylim(0.4, 1.0)
plt.show()

print("Brecha de sobreajuste (Train AUC - Test AUC):")
display(cv_summary[['model', 'train_roc_auc_mean', 'test_roc_auc_mean', 'auc_overfit_gap']])


## Parte 13 — Evaluación en profundidad de los finalistas
### Paso 51 y 52: Predicciones fuera de muestra, Matriz de Confusión y Curvas ROC / PR

In [ ]:
candidates = ml.get_candidates(42)
pipe_final = ml.build_pipeline(candidates['hgb'])

proba_cv = cross_val_predict(pipe_final, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
pred_cv = (proba_cv >= 0.5).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ConfusionMatrixDisplay.from_predictions(y_train, pred_cv, ax=axes[0], cmap='Blues')
axes[0].set_title('Matriz de Confusión (OOF CV)')

RocCurveDisplay.from_predictions(y_train, proba_cv, ax=axes[1], name='HGB')
axes[1].set_title('Curva ROC')

PrecisionRecallDisplay.from_predictions(y_train, proba_cv, ax=axes[2], name='HGB')
axes[2].set_title('Curva Precision-Recall')

plt.tight_layout()
plt.show()

print(classification_report(y_train, pred_cv))


### Paso 53: Calibración y elección del umbral

In [ ]:
prob_true, prob_pred = calibration_curve(y_train, proba_cv, n_bins=8)
brier = brier_score_loss(y_train, proba_cv)

plt.figure(figsize=(6, 4))
plt.plot(prob_pred, prob_true, marker='o', label=f'HGB (Brier={brier:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectamente calibrado')
plt.xlabel('Probabilidad predicha')
plt.ylabel('Fracción de positivos observada')
plt.title('Curva de Calibración')
plt.legend()
plt.show()

precisions, recalls, thresholds = precision_recall_curve(y_train, proba_cv)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[min(best_idx, len(thresholds) - 1)]

print(f"Brier score: {brier:.4f} (azar = 0.25, perfecto = 0)")
print(f"Umbral que maximiza F1 en validación: {best_threshold:.4f} (F1 máx = {f1_scores[best_idx]:.4f})")
print(f"F1 con umbral estándar 0.5: {f1_score(y_train, pred_cv):.4f}")


### Paso 54: Análisis de errores por subgrupos

In [ ]:
df_errors = X_train.copy()
df_errors['y_true'] = y_train
df_errors['proba'] = proba_cv
df_errors['pred'] = pred_cv
df_errors['error'] = df_errors['y_true'] != df_errors['pred']

print("Tasa de error por Sexo:")
display(df_errors.groupby('Sex')['error'].agg(['mean', 'count']))

print("\nTasa de error por Clase:")
display(df_errors.groupby('Pclass')['error'].agg(['mean', 'count']))

print("\n10 errores concretos con alta confianza:")
high_conf_errors = df_errors[df_errors['error']].sort_values(
    by='proba', key=lambda p: np.abs(p - 0.5), ascending=False
).head(10)
display(high_conf_errors[['Sex', 'Pclass', 'Age', 'Fare', 'TTitulo', 'y_true', 'proba', 'pred']])


### Paso 55: Curva de aprendizaje

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    pipe_final, X_train, y_train, cv=cv, train_sizes=np.linspace(0.2, 1.0, 5), scoring='roc_auc'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

plt.figure(figsize=(7, 4))
plt.plot(train_sizes, train_mean, 'o-', color='r', label='Entrenamiento')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='r')
plt.plot(train_sizes, val_mean, 'o-', color='g', label='Validación cruzada')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='g')
plt.title('Curva de Aprendizaje (ROC-AUC)')
plt.xlabel('Tamaño de muestra de entrenamiento')
plt.ylabel('ROC-AUC')
plt.legend()
plt.show()


## Parte 14 — Ajuste de hiperparámetros
### Pasos 56 a 58: Búsqueda con GridSearchCV

In [ ]:
param_grids = ml.get_param_grids()
grid_hgb = GridSearchCV(
    pipe_final, param_grids['hgb'], cv=cv, scoring='roc_auc', n_jobs=-1, refit=True
)
grid_hgb.fit(X_train, y_train)

print(f"Mejor ROC-AUC en CV: {grid_hgb.best_score_:.4f}")
print("Mejores parámetros encontrados:")
print(grid_hgb.best_params_)


## Parte 15 — Interpretación y ablación de variables
### Paso 59: Coeficientes de Regresión Logística y Odds Ratios

In [ ]:
pipe_lr = ml.build_pipeline(candidates['logreg']).fit(X_train, y_train)
feature_names_out = pipe_lr['prep'].get_feature_names_out()
coefs = pipe_lr['model'].coef_[0]

df_coefs = pd.DataFrame({
    'Feature': feature_names_out,
    'Coeficiente': coefs,
    'Odds_Ratio': np.exp(coefs)
}).sort_values(by='Coeficiente', ascending=False).reset_index(drop=True)

display(df_coefs)


### Paso 60: Importancia por permutación

In [ ]:
X_tr_sub, X_val_sub, y_tr_sub, y_val_sub = ml.split(X_train, y_train, test_size=0.25, random_state=42)
pipe_fit_sub = ml.build_pipeline(candidates['hgb']).fit(X_tr_sub, y_tr_sub)

perm_imp = permutation_importance(
    pipe_fit_sub, X_val_sub, y_val_sub, scoring='roc_auc', n_repeats=30, random_state=42
)

df_perm = pd.DataFrame({
    'Feature': X_val_sub.columns,
    'Importance_Mean': perm_imp.importances_mean,
    'Importance_Std': perm_imp.importances_std,
}).sort_values(by='Importance_Mean', ascending=False).reset_index(drop=True)

plt.figure(figsize=(8, 4))
sns.barplot(data=df_perm, x='Importance_Mean', y='Feature', palette='Blues_r')
plt.title('Importancia por Permutación (Validación, ROC-AUC)')
plt.xlabel('Caída en ROC-AUC al permutar la variable')
plt.show()


### Paso 61: Estudio de Ablación

In [ ]:
# Configuraciones de variables
configs = {
    "A) Base": (ml.NUM_FEATURES, ml.CAT_FEATURES),
    "B) Base + Cabina": (ml.NUM_FEATURES + ml.CABIN_NUM, ml.CAT_FEATURES + ml.CABIN_CAT),
    "C) Sin Titulo": (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != "TTitulo"]),
    "D) Sin Sexo": (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != "Sex"]),
}

ablation_results = []
for name, (num_f, cat_f) in configs.items():
    pipe_ab = ml.build_pipeline(candidates['hgb'], num_features=num_f, cat_features=cat_f)
    res_ab = ml.cross_validate(pipe_ab, X_train, y_train, cv=cv, scoring='roc_auc')
    ablation_results.append({
        "Configuracion": name,
        "ROC_AUC_Mean": float(res_ab['test_score'].mean()),
        "ROC_AUC_Std": float(res_ab['test_score'].std()),
    })

df_ablation = pd.DataFrame(ablation_results)
df_ablation['Delta_vs_Base'] = df_ablation['ROC_AUC_Mean'] - df_ablation.loc[0, 'ROC_AUC_Mean']
display(df_ablation)


## Parte 16 — Evaluación final honesta sobre el conjunto de prueba
### Pasos 62 y 63: Evaluación única con incertidumbre bootstrap

In [ ]:
# Ajuste del modelo elegido sobre todo X_train
num_final = ml.NUM_FEATURES + (ml.CABIN_NUM if ml.USE_CABIN_FEATURES else [])
cat_final = ml.CAT_FEATURES + (ml.CABIN_CAT if ml.USE_CABIN_FEATURES else [])

pipe_eval_test = ml.build_pipeline(candidates[ml.FINAL_ESTIMATOR], num_features=num_final, cat_features=cat_final)
pipe_eval_test.set_params(**ml.FINAL_PARAMS)
pipe_eval_test.fit(X_train, y_train)

test_metrics = ml.evaluate_on_test(pipe_eval_test, X_test, y_test, threshold=ml.THRESHOLD)
print("--- Métricas en el Conjunto de Prueba ---")
print(test_metrics)

# Bootstrap 95% CI (1000 remuestreos)
proba_test = pipe_eval_test.predict_proba(X_test)[:, 1]
rng = np.random.default_rng(42)
n_test = len(y_test)
boot_aucs, boot_accs = [], []

for _ in range(1000):
    idx = rng.choice(n_test, size=n_test, replace=True)
    y_b = y_test.iloc[idx]
    if len(np.unique(y_b)) < 2:
        continue
    p_b = proba_test[idx]
    boot_aucs.append(ml.roc_auc_score(y_b, p_b))
    boot_accs.append(ml.accuracy_score(y_b, (p_b >= ml.THRESHOLD).astype(int)))

ci_auc = (np.percentile(boot_aucs, 2.5), np.percentile(boot_aucs, 97.5))
ci_acc = (np.percentile(boot_accs, 2.5), np.percentile(boot_accs, 97.5))

print(f"ROC-AUC Test: {test_metrics['roc_auc']:.4f} (IC 95%: [{ci_auc[0]:.4f}, {ci_auc[1]:.4f}])")
print(f"Accuracy Test: {test_metrics['accuracy']:.4f} (IC 95%: [{ci_acc[0]:.4f}, {ci_acc[1]:.4f}])")


## Parte 17 — Modelo definitivo y prueba de humo
### Paso 65 a 69: Carga del modelo guardado y prueba con datos sin etiqueta

In [ ]:
# Cargar el modelo guardado desde disco
pipe_loaded, metadata_loaded = ml.load_model(MODELS / f"{ml.MODEL_NAME}.joblib")
print("Modelo cargado exitosamente. Metadatos registrados:")
print(f"Estimador: {metadata_loaded['estimator']}, Python: {metadata_loaded['python_version']}, Sklearn: {metadata_loaded['sklearn_version']}")

# Prueba de humo con datos sin etiqueta (titanic_unlabeled.csv)
df_unlabeled = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
print(f"Datos sin etiqueta leídos: {df_unlabeled.shape[0]} filas.")

# Preprocesar con las transformaciones deterministas de Silver
unlabeled_processed = de.build_silver(df_unlabeled)
X_unlabeled = unlabeled_processed[ml.CANDIDATE_COLUMNS].copy()
X_unlabeled['Age'] = unlabeled_processed['AgeOriginal']

predictions_new = ml.predict(pipe_loaded, X_unlabeled, threshold=ml.THRESHOLD)
print("Predicciones para los pasajeros nuevos:")
print(f"¿Hay valores NaN en las predicciones? {predictions_new.isna().any().to_dict()}")
print(f"Distribución de supervivientes predichos:\n{predictions_new['pred'].value_counts(normalize=True).round(3)}")
display(predictions_new.head(10))
